# Benchmark Evaluation Runner

This notebook is the artifact-only evaluation consumer.
It loads a completed training run from `ml_model/results/benchmarks/` and computes comparisons without retraining.

Validation-calibrated ECE is reported as an in-sample diagnostic only.

In [ ]:
import os
import sys
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoTokenizer


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml_model.preprocessing.dataset_io import (
    DEFAULT_RUNS_DIR,
    evaluation_dir,
    latest_run_dir,
    load_data_splits,
    load_json,
    load_numpy_artifacts,
    resolve_data_dir,
    save_csv,
    save_json,
)
from ml_model.evaluation.metrics import (
    DEFAULT_ROBUSTNESS_PERTURBATIONS,
    apply_perturbation_batch,
    collect_logits_from_texts,
    confidence_band_summary_frame,
    evaluate_from_logits,
    per_class_recall_at_threshold_frame,
    robustness_failure_examples_frame,
    robustness_retention_row,
    save_confusion_matrix_artifacts,
    save_reliability_diagram_artifacts,
    threshold_security_summary,
    top_label_calibration_frame,
)
from ml_model.training.model_factory import build_model

In [ ]:
DATASET_VERSION = "v3_907k_cleaned"
ECE_N_BINS = 15
CONFIDENCE_THRESHOLDS = [0.5, 0.7, 0.8, 0.9]
ROBUSTNESS_MAX_EXAMPLES = 25

EVAL_SMOKE_MODE = os.getenv("BENCHMARK_EVAL_SMOKE_MODE", "").strip().lower() in {"1", "true", "yes", "on"}
EVAL_ENABLE_ROBUSTNESS = os.getenv("BENCHMARK_EVAL_ENABLE_ROBUSTNESS", "1").strip().lower() in {
    "1",
    "true",
    "yes",
    "on",
}
ROBUSTNESS_PERTURBATIONS = dict(DEFAULT_ROBUSTNESS_PERTURBATIONS)
if EVAL_SMOKE_MODE:
    EVAL_ENABLE_ROBUSTNESS = False
if not EVAL_ENABLE_ROBUSTNESS:
    ROBUSTNESS_PERTURBATIONS = {}

run_dir_override = os.getenv("BENCHMARK_RUN_DIR", "").strip()
if run_dir_override:
    RUN_DIR = Path(run_dir_override).expanduser().resolve()
else:
    RUN_DIR = latest_run_dir(base_dir=DEFAULT_RUNS_DIR, dataset_version=DATASET_VERSION)

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run directory does not exist: {RUN_DIR}")

EVALUATION_OUTPUT_DIR = evaluation_dir(RUN_DIR)
MANIFEST_PATH = RUN_DIR / "run_manifest.json"

manifest = load_json(MANIFEST_PATH) if MANIFEST_PATH.exists() else {}
DATASET_VERSION = manifest.get("dataset_version", DATASET_VERSION)
LABEL_NAMES = manifest.get("label_names", [])

manifest_model_keys = manifest.get("run_model_keys")
if isinstance(manifest_model_keys, (list, tuple, set)):
    model_keys = [str(key) for key in manifest_model_keys]
else:
    model_keys = []
if not model_keys:
    model_keys = sorted(p.name for p in RUN_DIR.iterdir() if p.is_dir() and (p / "loss_variant_aggregates.csv").exists())
if not model_keys:
    model_keys = sorted(p.name for p in RUN_DIR.iterdir() if p.is_dir())

if not model_keys:
    raise FileNotFoundError(f"No model artifacts found in run directory: {RUN_DIR}")

DATA_DIR = resolve_data_dir(DATASET_VERSION)
evaluated_test_rows_path = RUN_DIR / "evaluated_test_rows.csv"
if evaluated_test_rows_path.exists():
    df_test = pd.read_csv(evaluated_test_rows_path)
else:
    _, _, df_test = load_data_splits(DATA_DIR, manifest.get("text_col", "combined_payload"), manifest.get("label_col", "final_label"))

print(f"Run dir            : {RUN_DIR}")
print(f"Evaluation dir     : {EVALUATION_OUTPUT_DIR}")
print(f"Data dir           : {DATA_DIR}")
print(f"Model keys         : {model_keys}")
print(f"Manifest loaded    : {MANIFEST_PATH.exists()}")
print(f"Label names loaded : {LABEL_NAMES}")
print(f"Evaluation smoke   : {EVAL_SMOKE_MODE}")
print(f"Robustness enabled : {EVAL_ENABLE_ROBUSTNESS}")

In [ ]:
EVAL_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EVAL_CUDA_BF16 = torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()
TEXT_COL = manifest.get("text_col", "combined_payload")


def get_eval_autocast_context():
    if EVAL_DEVICE.type == "cuda" and EVAL_CUDA_BF16:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def selected_loss_key_for_model(model_key: str) -> str:
    model_selection_manifest = manifest.get("model_selection_manifest", {})
    if model_key in model_selection_manifest:
        selected = model_selection_manifest[model_key].get("selected_loss_key")
        if selected:
            return str(selected)

    model_dir = RUN_DIR / model_key
    aggregate_path = model_dir / "loss_variant_aggregates.csv"
    if aggregate_path.exists():
        aggregate_df = pd.read_csv(aggregate_path)
        if "val_macro_f1_mean" in aggregate_df.columns:
            top_row = aggregate_df.sort_values(by="val_macro_f1_mean", ascending=False).iloc[0]
            return str(top_row["loss_key"])
        return str(aggregate_df.iloc[0]["loss_key"])

    fallback_loss_dirs = sorted(p for p in model_dir.glob("loss_*") if p.is_dir())
    if fallback_loss_dirs:
        return fallback_loss_dirs[0].name.replace("loss_", "", 1)

    raise FileNotFoundError(f"Could not determine selected loss key for model {model_key}")


def seed_dirs_for_model(model_key: str, loss_key: str) -> list[Path]:
    variant_dir = RUN_DIR / model_key / f"loss_{loss_key}"
    if not variant_dir.exists():
        raise FileNotFoundError(f"Missing variant directory: {variant_dir}")
    seed_dirs = sorted([p for p in variant_dir.glob("seed_*") if p.is_dir()])
    if not seed_dirs:
        raise FileNotFoundError(f"No seed directories found in {variant_dir}")
    return seed_dirs


def parse_seed(seed_dir: Path) -> int:
    try:
        return int(seed_dir.name.split("_")[-1])
    except Exception:
        return -1


def load_seed_artifacts(seed_dir: Path) -> tuple[dict, dict, float, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    summary = load_json(seed_dir / "summary_metrics.json") if (seed_dir / "summary_metrics.json").exists() else {}
    config = load_json(seed_dir / "config_metadata.json") if (seed_dir / "config_metadata.json").exists() else {}

    calibration_path = seed_dir / "calibration.json"
    temperature = 1.0
    if calibration_path.exists():
        calibration = load_json(calibration_path)
        temperature = float(calibration.get("temperature", 1.0))

    val_outputs = load_numpy_artifacts(seed_dir / "validation_outputs.npz")
    test_outputs = load_numpy_artifacts(seed_dir / "test_outputs.npz")

    val_logits = val_outputs["logits"]
    val_labels = val_outputs["labels"].astype(np.int64)
    test_logits = test_outputs["logits"]
    test_labels = test_outputs["labels"].astype(np.int64)

    return summary, config, temperature, val_logits, val_labels, test_logits, test_labels


def write_seed_reports(
    model_key: str,
    loss_key: str,
    seed: int,
    test_labels: np.ndarray,
    test_uncal: dict,
    test_cal: dict,
):
    prefix = f"{model_key}_{loss_key}_seed{int(seed):04d}"

    save_confusion_matrix_artifacts(
        labels=test_labels,
        preds=test_uncal["preds"],
        label_names=LABEL_NAMES,
        csv_path=EVALUATION_OUTPUT_DIR / f"{prefix}_confusion_matrix.csv",
        png_path=EVALUATION_OUTPUT_DIR / f"{prefix}_confusion_matrix.png",
        title=f"{prefix} confusion matrix",
    )

    save_reliability_diagram_artifacts(
        probs=test_uncal["probs"],
        labels=test_labels,
        csv_path=EVALUATION_OUTPUT_DIR / f"{prefix}_reliability_uncalibrated.csv",
        png_path=EVALUATION_OUTPUT_DIR / f"{prefix}_reliability_uncalibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{prefix} reliability (uncalibrated)",
    )
    save_reliability_diagram_artifacts(
        probs=test_cal["probs"],
        labels=test_labels,
        csv_path=EVALUATION_OUTPUT_DIR / f"{prefix}_reliability_calibrated.csv",
        png_path=EVALUATION_OUTPUT_DIR / f"{prefix}_reliability_calibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{prefix} reliability (calibrated)",
    )

    top_uncal_df = top_label_calibration_frame(test_uncal["probs"], test_labels, LABEL_NAMES)
    top_cal_df = top_label_calibration_frame(test_cal["probs"], test_labels, LABEL_NAMES)
    save_csv(top_uncal_df, EVALUATION_OUTPUT_DIR / f"{prefix}_top_label_uncalibrated.csv", index=False)
    save_csv(top_cal_df, EVALUATION_OUTPUT_DIR / f"{prefix}_top_label_calibrated.csv", index=False)

    security_views = threshold_security_summary(
        labels=test_labels,
        preds=test_uncal["preds"],
        probs=test_uncal["probs"],
        label_names=LABEL_NAMES,
        normal_label="Normal",
    )

    save_csv(
        security_views["attack_to_normal_by_class"],
        EVALUATION_OUTPUT_DIR / f"{prefix}_attack_to_normal_fn.csv",
        index=False,
    )
    save_csv(
        confidence_band_summary_frame(test_labels, test_uncal["preds"], test_uncal["probs"]),
        EVALUATION_OUTPUT_DIR / f"{prefix}_confidence_band_summary.csv",
        index=False,
    )
    save_csv(
        per_class_recall_at_threshold_frame(
            labels=test_labels,
            preds=test_uncal["preds"],
            probs=test_uncal["probs"],
            label_names=LABEL_NAMES,
            thresholds=CONFIDENCE_THRESHOLDS,
        ),
        EVALUATION_OUTPUT_DIR / f"{prefix}_per_class_recall_at_threshold.csv",
        index=False,
    )
    save_json(
        EVALUATION_OUTPUT_DIR / f"{prefix}_security_summary.json",
        {
            "normal_false_positive": security_views["normal_false_positive"],
            "attack_escape_total": security_views["attack_escape_total"],
            "confidence_thresholds": CONFIDENCE_THRESHOLDS,
            "confidence_band_definition": {
                "LOW": "[0.0, 0.5)",
                "MEDIUM": "[0.5, 0.8)",
                "HIGH": "[0.8, 1.0]",
            },
        },
    )

    return security_views


def run_robustness_suite(
    model_key: str,
    loss_key: str,
    seed: int,
    config: dict,
    seed_dir: Path,
    baseline_preds: np.ndarray,
    labels: np.ndarray,
) -> list[dict]:
    checkpoint_dir = seed_dir / "checkpoint"
    checkpoint_candidates = sorted(checkpoint_dir.glob("best_*.pt"))
    if not checkpoint_candidates:
        return []

    checkpoint = torch.load(checkpoint_candidates[0], map_location=EVAL_DEVICE)
    model_cfg = {
        "model_id": config.get("model_id", checkpoint.get("cfg", {}).get("model_id")),
        "architecture": config.get("architecture", checkpoint.get("cfg", {}).get("architecture")),
        "dropout_prob": checkpoint.get("cfg", {}).get("dropout_prob", 0.2),
        "head_hidden_dim": checkpoint.get("cfg", {}).get("head_hidden_dim", 256),
        "activation": checkpoint.get("cfg", {}).get("activation", "gelu"),
        "rnn_hidden_dim": checkpoint.get("cfg", {}).get("rnn_hidden_dim", 256),
        "rnn_layers": checkpoint.get("cfg", {}).get("rnn_layers", 1),
        "bidirectional": checkpoint.get("cfg", {}).get("bidirectional", True),
        "attn_dim": checkpoint.get("cfg", {}).get("attn_dim", 128),
        "num_filters": checkpoint.get("cfg", {}).get("num_filters", 128),
        "kernel_sizes": checkpoint.get("cfg", {}).get("kernel_sizes", [3, 5, 7]),
    }
    max_len = int(config.get("max_seq_len", checkpoint.get("cfg", {}).get("max_seq_len", 128)))
    batch_size = int(config.get("training_hyperparameters", {}).get("per_device_train_batch_size", 64))

    model = build_model(model_cfg, num_classes=len(LABEL_NAMES), device=EVAL_DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_id"], use_fast=True)
    base_texts = df_test[TEXT_COL].astype(str).head(len(labels)).tolist()

    rows = []
    for perturbation_key in ROBUSTNESS_PERTURBATIONS:
        perturbed_texts = apply_perturbation_batch(base_texts, perturbation_key)
        perturbed_logits = collect_logits_from_texts(
            model=model,
            tokenizer=tokenizer,
            texts=perturbed_texts,
            device=EVAL_DEVICE,
            max_len=max_len,
            batch_size=batch_size,
            autocast_context_fn=get_eval_autocast_context,
        )
        perturbed_metrics = evaluate_from_logits(perturbed_logits, labels, n_bins=ECE_N_BINS)

        row = robustness_retention_row(
            labels=labels,
            baseline_preds=baseline_preds,
            perturbed_preds=perturbed_metrics["preds"],
            perturbation_key=perturbation_key,
        )
        row.update(
            {
                "model_key": model_key,
                "loss_key": loss_key,
                "seed": int(seed),
                "description": ROBUSTNESS_PERTURBATIONS[perturbation_key],
            }
        )
        rows.append(row)

        failures_df = robustness_failure_examples_frame(
            original_texts=base_texts,
            perturbed_texts=perturbed_texts,
            labels=labels,
            baseline_preds=baseline_preds,
            perturbed_preds=perturbed_metrics["preds"],
            label_names=LABEL_NAMES,
            perturbation_key=perturbation_key,
            max_examples=ROBUSTNESS_MAX_EXAMPLES,
        )
        if not failures_df.empty:
            save_csv(
                failures_df,
                EVALUATION_OUTPUT_DIR
                / f"{model_key}_{loss_key}_seed{int(seed):04d}_{perturbation_key}_robustness_failures.csv",
                index=False,
            )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return rows


comparison_rows = []
robustness_rows_all = []

for model_key in model_keys:
    loss_key = selected_loss_key_for_model(model_key)
    seed_dirs = seed_dirs_for_model(model_key, loss_key)
    model_seed_rows = []
    model_robustness_rows = []

    for seed_dir in seed_dirs:
        seed = parse_seed(seed_dir)
        summary, config, temperature, val_logits, val_labels, test_logits, test_labels = load_seed_artifacts(seed_dir)

        val_uncal = evaluate_from_logits(val_logits, val_labels, n_bins=ECE_N_BINS)
        test_uncal = evaluate_from_logits(test_logits, test_labels, n_bins=ECE_N_BINS)
        val_cal = evaluate_from_logits(val_logits / temperature, val_labels, n_bins=ECE_N_BINS)
        test_cal = evaluate_from_logits(test_logits / temperature, test_labels, n_bins=ECE_N_BINS)

        security_views = write_seed_reports(
            model_key=model_key,
            loss_key=loss_key,
            seed=seed,
            test_labels=test_labels,
            test_uncal=test_uncal,
            test_cal=test_cal,
        )

        model_seed_rows.append(
            {
                "model_key": model_key,
                "loss_key": loss_key,
                "seed": int(seed),
                "architecture": summary.get("architecture", config.get("architecture")),
                "architecture_family": summary.get("architecture_family", "unknown"),
                "head_type": summary.get("head_type", "unknown"),
                "experiment_phase": summary.get("experiment_phase", "unknown"),
                "temperature": float(temperature),
                "trainable_params": float(summary.get("trainable_params", 0)),
                "val_accuracy": float(val_uncal["accuracy"]),
                "val_macro_f1": float(val_uncal["macro_f1"]),
                "val_weighted_f1": float(val_uncal["weighted_f1"]),
                "val_ece_uncalibrated": float(val_uncal["ece"]),
                "val_ece_calibrated": float(val_cal["ece"]),
                "val_nll_uncalibrated": float(val_uncal["nll"]),
                "val_nll_calibrated": float(val_cal["nll"]),
                "test_accuracy": float(test_uncal["accuracy"]),
                "test_macro_f1": float(test_uncal["macro_f1"]),
                "test_weighted_f1": float(test_uncal["weighted_f1"]),
                "test_ece_uncalibrated": float(test_uncal["ece"]),
                "test_ece_calibrated": float(test_cal["ece"]),
                "test_nll_uncalibrated": float(test_uncal["nll"]),
                "test_nll_calibrated": float(test_cal["nll"]),
                "normal_false_positive_rate": float(
                    security_views["normal_false_positive"]["normal_false_positive_rate"]
                ),
                "attack_escape_rate": float(security_views["attack_escape_total"]["attack_escape_rate"]),
                "inference_latency_ms": float(summary.get("inference_latency_mean_ms", np.nan)),
                "model_size_mb": float(summary.get("model_size_mb", np.nan)),
            }
        )

        robustness_rows = run_robustness_suite(
            model_key=model_key,
            loss_key=loss_key,
            seed=seed,
            config=config,
            seed_dir=seed_dir,
            baseline_preds=test_uncal["preds"],
            labels=test_labels,
        )
        model_robustness_rows.extend(robustness_rows)

    seed_df = pd.DataFrame(model_seed_rows).sort_values(by="seed").reset_index(drop=True)
    save_csv(seed_df, EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_seed_metrics.csv", index=False)

    numeric_cols = [
        col
        for col in seed_df.columns
        if pd.api.types.is_numeric_dtype(seed_df[col]) and col not in {"seed"}
    ]
    aggregate = {}
    for col in numeric_cols:
        values = np.asarray(seed_df[col].to_numpy(dtype=np.float64), dtype=np.float64)
        aggregate[f"{col}_mean"] = float(np.mean(values))
        aggregate[f"{col}_std"] = float(np.std(values, ddof=0))

    model_row = {
        "model_key": model_key,
        "loss_key": loss_key,
        "n_seeds": int(seed_df.shape[0]),
        "architecture": str(seed_df.iloc[0]["architecture"]),
        "architecture_family": str(seed_df.iloc[0]["architecture_family"]),
        "head_type": str(seed_df.iloc[0]["head_type"]),
        "experiment_phase": str(seed_df.iloc[0]["experiment_phase"]),
        **aggregate,
    }
    comparison_rows.append(model_row)

    if model_robustness_rows:
        model_robustness_df = pd.DataFrame(model_robustness_rows)
        save_csv(model_robustness_df, EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_robustness.csv", index=False)
        robustness_rows_all.extend(model_robustness_rows)

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by=["experiment_phase", "model_key"],
    ascending=[True, True],
).reset_index(drop=True)

summary_cols = [
    "model_key",
    "experiment_phase",
    "loss_key",
    "architecture",
    "head_type",
    "n_seeds",
    "val_macro_f1_mean",
    "val_macro_f1_std",
    "test_macro_f1_mean",
    "test_macro_f1_std",
    "test_ece_uncalibrated_mean",
    "test_ece_calibrated_mean",
    "test_nll_uncalibrated_mean",
    "test_nll_calibrated_mean",
    "normal_false_positive_rate_mean",
    "attack_escape_rate_mean",
    "inference_latency_ms_mean",
    "model_size_mb_mean",
]

display(
    comparison_df[summary_cols].style.format(
        {
            "val_macro_f1_mean": "{:.4f}",
            "val_macro_f1_std": "{:.4f}",
            "test_macro_f1_mean": "{:.4f}",
            "test_macro_f1_std": "{:.4f}",
            "test_ece_uncalibrated_mean": "{:.4f}",
            "test_ece_calibrated_mean": "{:.4f}",
            "test_nll_uncalibrated_mean": "{:.4f}",
            "test_nll_calibrated_mean": "{:.4f}",
            "normal_false_positive_rate_mean": "{:.4f}",
            "attack_escape_rate_mean": "{:.4f}",
            "inference_latency_ms_mean": "{:.3f}",
            "model_size_mb_mean": "{:.2f}",
        }
    )
)

robustness_df = pd.DataFrame(robustness_rows_all)
if not robustness_df.empty:
    save_csv(robustness_df, EVALUATION_OUTPUT_DIR / "robustness_summary.csv", index=False)
    robustness_view = (
        robustness_df.groupby(["model_key", "loss_key", "perturbation"], as_index=False)
        .agg(
            accuracy_retention_mean=("accuracy_retention", "mean"),
            macro_f1_retention_mean=("macro_f1_retention", "mean"),
            accuracy_drop_mean=("accuracy_drop", "mean"),
            macro_f1_drop_mean=("macro_f1_drop", "mean"),
        )
        .sort_values(by=["model_key", "perturbation"])
        .reset_index(drop=True)
    )
    save_csv(robustness_view, EVALUATION_OUTPUT_DIR / "robustness_aggregated.csv", index=False)
else:
    robustness_view = pd.DataFrame()

print("Evaluation tables prepared.")
print("Test metrics are reported as final evidence only; no test-set winner selection is performed.")

In [ ]:
if comparison_df.empty:
    raise RuntimeError("No comparison rows were produced from run artifacts.")

validation_ranking_df = comparison_df.sort_values(
    by=["val_macro_f1_mean"],
    ascending=[False],
).reset_index(drop=True)

display(
    validation_ranking_df[
        [
            "model_key",
            "loss_key",
            "experiment_phase",
            "val_macro_f1_mean",
            "val_macro_f1_std",
            "val_ece_calibrated_mean",
            "val_nll_calibrated_mean",
        ]
    ].style.format(
        {
            "val_macro_f1_mean": "{:.4f}",
            "val_macro_f1_std": "{:.4f}",
            "val_ece_calibrated_mean": "{:.4f}",
            "val_nll_calibrated_mean": "{:.4f}",
        }
    )
)

print("Validation ranking is shown for auditability only.")
print("Validation-calibrated ECE is diagnostic-only and is not used in ranking.")
print("This notebook does not declare a best model from test metrics.")

In [ ]:
save_csv(comparison_df, EVALUATION_OUTPUT_DIR / "model_comparison.csv", index=False)

if not robustness_view.empty:
    save_csv(robustness_view, EVALUATION_OUTPUT_DIR / "robustness_aggregated.csv", index=False)

summary_payload = {
    "run_dir": str(RUN_DIR),
    "evaluation_output_dir": str(EVALUATION_OUTPUT_DIR),
    "dataset_version": DATASET_VERSION,
    "selection_rule": manifest.get("model_selection_rule", "validation-driven selection"),
    "reporting_policy": "Test metrics are final evidence outputs and are not used for winner selection.",
    "confidence_bands": {
        "LOW": "[0.0, 0.5)",
        "MEDIUM": "[0.5, 0.8)",
        "HIGH": "[0.8, 1.0]",
    },
    "model_count": int(comparison_df.shape[0]),
    "robustness_rows": int(robustness_df.shape[0]) if not robustness_df.empty else 0,
    "generated_outputs": [
        "model_comparison.csv",
        "*_seed_metrics.csv",
        "*_security_summary.json",
        "*_reliability_*.csv/.png",
        "robustness_summary.csv",
        "robustness_aggregated.csv",
    ],
}
save_json(EVALUATION_OUTPUT_DIR / "evaluation_summary.json", summary_payload)

print("Saved:")
print(f"- {EVALUATION_OUTPUT_DIR / 'model_comparison.csv'}")
print(f"- {EVALUATION_OUTPUT_DIR / 'evaluation_summary.json'}")
if not robustness_view.empty:
    print(f"- {EVALUATION_OUTPUT_DIR / 'robustness_summary.csv'}")
    print(f"- {EVALUATION_OUTPUT_DIR / 'robustness_aggregated.csv'}")
print("Per-seed calibration/threshold/robustness artifacts were emitted in evaluation output.")